# Этап 3. Парсинг и первичный анализ датасета

Ноутбук для Google Colab. Выполняет:
- анализ структуры датасета;
- проверку изображений и XML-аннотаций;
- построение таблицы разметки;
- выделение `single-class` выборки для классификации;
- формирование `train / val / test split`;
- сохранение итоговых CSV-файлов.

## Как использовать

1. Загрузите папку проекта в Google Drive.
2. Убедитесь, что датасет лежит по пути `.../MyProject1/DB`.
3. Запустите ноутбук сверху вниз.
4. Итоговые CSV будут сохранены в `stage3_outputs_colab`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from collections import Counter
import json
import random
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/MyProject1')
DATASET_ROOT = PROJECT_ROOT / 'DB'
IMAGES_ROOT = DATASET_ROOT / 'images' / 'images'
XML_ROOT = DATASET_ROOT / 'label' / 'label'
OUTPUT_DIR = PROJECT_ROOT / 'stage3_outputs_colab'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

assert IMAGES_ROOT.exists(), f'Не найдена папка с изображениями: {IMAGES_ROOT}'
assert XML_ROOT.exists(), f'Не найдена папка с XML: {XML_ROOT}'

CLASS_MAPPING = {
    '1_chongkong': {'class_id': 1, 'class_name': 'punching_hole'},
    '2_hanfeng': {'class_id': 2, 'class_name': 'welding_line'},
    '3_yueyawan': {'class_id': 3, 'class_name': 'crescent_gap'},
    '4_shuiban': {'class_id': 4, 'class_name': 'water_spot'},
    '5_youban': {'class_id': 5, 'class_name': 'oil_spot'},
    '6_siban': {'class_id': 6, 'class_name': 'silk_spot'},
    '7_yiwu': {'class_id': 7, 'class_name': 'inclusion'},
    '8_yahen': {'class_id': 8, 'class_name': 'rolled_pit'},
    '9_zhehen': {'class_id': 9, 'class_name': 'crease'},
    '10_yaozhed': {'class_id': 10, 'class_name': 'waist_folding'},
}

NORMALIZE_LABELS = {
    '10_yaozhe': '10_yaozhed',
}

print('PROJECT_ROOT =', PROJECT_ROOT)
print('IMAGES_ROOT  =', IMAGES_ROOT)
print('XML_ROOT     =', XML_ROOT)
print('OUTPUT_DIR   =', OUTPUT_DIR)

In [ ]:
def normalize_label(label: str) -> str:
    label = label.strip()
    return NORMALIZE_LABELS.get(label, label)


def parse_xml_labels(xml_path: Path) -> list[str]:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    labels = []
    for node in root.findall('.//object/name'):
        labels.append(normalize_label(node.text or ''))
    return sorted(set(labels))


def read_image_size(image_path: Path) -> tuple[int, int]:
    with Image.open(image_path) as img:
        width, height = img.size
    return width, height


def build_xml_index(xml_root: Path) -> dict[str, Path]:
    xml_files = list(xml_root.glob('*.xml'))
    xml_index = {}
    duplicates = []
    for xml_file in xml_files:
        if xml_file.stem in xml_index:
            duplicates.append(xml_file.stem)
        xml_index[xml_file.stem] = xml_file
    return xml_index, sorted(set(duplicates))


def collect_image_records(images_root: Path, xml_root: Path) -> pd.DataFrame:
    xml_index, duplicate_xml_stems = build_xml_index(xml_root)
    image_paths = sorted([
        path for path in images_root.rglob('*')
        if path.is_file() and path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}
    ])

    basename_counter = Counter(path.stem for path in image_paths)
    duplicate_image_stems = {stem for stem, count in basename_counter.items() if count > 1}

    records = []
    for image_path in image_paths:
        xml_path = xml_index.get(image_path.stem)
        folder_name = image_path.parent.name

        broken_image = False
        width = None
        height = None
        try:
            width, height = read_image_size(image_path)
        except Exception:
            broken_image = True

        labels = []
        broken_xml = False
        if xml_path is not None:
            try:
                labels = parse_xml_labels(xml_path)
            except Exception:
                broken_xml = True

        records.append({
            'image_path': str(image_path),
            'image_name': image_path.name,
            'image_stem': image_path.stem,
            'source_folder': folder_name,
            'xml_path': str(xml_path) if xml_path else None,
            'xml_exists': xml_path is not None,
            'xml_labels': labels,
            'label_count': len(labels),
            'duplicate_image_stem': image_path.stem in duplicate_image_stems,
            'broken_image': broken_image,
            'broken_xml': broken_xml,
            'width': width,
            'height': height,
        })

    records_df = pd.DataFrame(records)
    records_df.attrs['duplicate_xml_stems'] = duplicate_xml_stems
    return records_df


def split_clean_dataset(clean_df: pd.DataFrame, random_state: int = 42):
    train_df, temp_df = train_test_split(
        clean_df,
        test_size=(1.0 - TRAIN_SIZE),
        stratify=clean_df['class_id'],
        random_state=random_state,
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df['class_id'],
        random_state=random_state,
    )

    train_df = train_df.assign(split='train').copy()
    val_df = val_df.assign(split='val').copy()
    test_df = test_df.assign(split='test').copy()
    return train_df, val_df, test_df


def show_class_examples(clean_df: pd.DataFrame, samples_per_class: int = 1, image_size=(10, 4)):
    classes = clean_df[['class_name', 'class_id']].drop_duplicates().sort_values('class_id')
    fig, axes = plt.subplots(len(classes), samples_per_class, figsize=(image_size[0] * samples_per_class, image_size[1] * len(classes)))
    if len(classes) == 1 and samples_per_class == 1:
        axes = [[axes]]
    elif len(classes) == 1:
        axes = [axes]
    elif samples_per_class == 1:
        axes = [[ax] for ax in axes]

    for row_idx, (_, class_row) in enumerate(classes.iterrows()):
        subset = clean_df[clean_df['class_name'] == class_row['class_name']].head(samples_per_class)
        for col_idx in range(samples_per_class):
            ax = axes[row_idx][col_idx]
            ax.axis('off')
            if col_idx < len(subset):
                img = Image.open(subset.iloc[col_idx]['image_path'])
                ax.imshow(img, cmap='gray')
                ax.set_title(f"{class_row['class_name']}\nID={class_row['class_id']}")
    plt.tight_layout()
    plt.show()

In [ ]:
records_df = collect_image_records(IMAGES_ROOT, XML_ROOT)
duplicate_xml_stems = records_df.attrs.get('duplicate_xml_stems', [])

print('Всего изображений           :', len(records_df))
print('XML есть                    :', int(records_df['xml_exists'].sum()))
print('Изображений без XML         :', int((~records_df['xml_exists']).sum()))
print('Битых изображений           :', int(records_df['broken_image'].sum()))
print('Битых XML                   :', int(records_df['broken_xml'].sum()))
print('Дубликатов image basename   :', int(records_df['duplicate_image_stem'].sum()))
print('Дубликатов xml basename     :', len(duplicate_xml_stems))

records_df.head()

In [ ]:
size_summary = (
    records_df[['width', 'height']]
    .dropna()
    .value_counts()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

folder_summary = (
    records_df.groupby('source_folder')
    .size()
    .reset_index(name='image_count')
    .sort_values('image_count', ascending=False)
)

label_count_summary = (
    records_df.groupby('label_count')
    .size()
    .reset_index(name='image_count')
    .sort_values('label_count')
)

display(folder_summary)
display(size_summary)
display(label_count_summary)

In [ ]:
clean_df = records_df[
    records_df['xml_exists']
    & (~records_df['broken_image'])
    & (~records_df['broken_xml'])
    & (~records_df['duplicate_image_stem'])
    & (records_df['label_count'] == 1)
].copy()

clean_df['xml_label'] = clean_df['xml_labels'].apply(lambda labels: labels[0])
clean_df = clean_df[clean_df['xml_label'].isin(CLASS_MAPPING)].copy()
clean_df['class_id'] = clean_df['xml_label'].apply(lambda label: CLASS_MAPPING[label]['class_id'])
clean_df['class_name'] = clean_df['xml_label'].apply(lambda label: CLASS_MAPPING[label]['class_name'])
clean_df['folder_matches_xml'] = clean_df.apply(
    lambda row: row['source_folder'].replace(' ', '_') == row['class_name'],
    axis=1,
)

excluded_df = records_df.copy()
excluded_df['reason'] = 'kept'
excluded_df.loc[~excluded_df['xml_exists'], 'reason'] = 'missing_xml'
excluded_df.loc[excluded_df['broken_image'], 'reason'] = 'broken_image'
excluded_df.loc[excluded_df['broken_xml'], 'reason'] = 'broken_xml'
excluded_df.loc[excluded_df['duplicate_image_stem'], 'reason'] = 'duplicate_image_stem'
excluded_df.loc[excluded_df['label_count'] > 1, 'reason'] = 'multi_class'
excluded_df.loc[(excluded_df['label_count'] == 1) & (~excluded_df['xml_labels'].apply(lambda x: x[0] if len(x) == 1 else None).isin(CLASS_MAPPING)), 'reason'] = 'unknown_label'

excluded_df = excluded_df[excluded_df['reason'] != 'kept'].copy()
excluded_df['xml_labels'] = excluded_df['xml_labels'].apply(lambda labels: '|'.join(labels))

print('Single-class для классификации:', len(clean_df))
print('Исключено записей             :', len(excluded_df))

clean_df[['image_path', 'class_name', 'class_id', 'xml_label', 'source_folder', 'folder_matches_xml']].head()

In [ ]:
class_summary_df = (
    clean_df.groupby(['class_id', 'class_name'])
    .agg(
        image_count=('image_path', 'count'),
        folder_match_count=('folder_matches_xml', 'sum'),
        folder_mismatch_count=('folder_matches_xml', lambda s: int((~s).sum())),
    )
    .reset_index()
    .sort_values('class_id')
)

display(class_summary_df)

plt.figure(figsize=(12, 5))
plt.bar(class_summary_df['class_name'], class_summary_df['image_count'])
plt.xticks(rotation=45, ha='right')
plt.title('Распределение single-class изображений по классам')
plt.ylabel('Количество изображений')
plt.tight_layout()
plt.show()

In [ ]:
train_df, val_df, test_df = split_clean_dataset(clean_df, random_state=RANDOM_SEED)
split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

split_summary_df = (
    split_df.groupby(['split', 'class_id', 'class_name'])
    .size()
    .reset_index(name='image_count')
    .sort_values(['split', 'class_id'])
)

print('train:', len(train_df))
print('val  :', len(val_df))
print('test :', len(test_df))
display(split_summary_df)

In [ ]:
show_class_examples(clean_df, samples_per_class=1)

In [ ]:
clean_manifest_path = OUTPUT_DIR / 'clean_manifest.csv'
excluded_manifest_path = OUTPUT_DIR / 'excluded_manifest.csv'
train_manifest_path = OUTPUT_DIR / 'train_manifest.csv'
val_manifest_path = OUTPUT_DIR / 'val_manifest.csv'
test_manifest_path = OUTPUT_DIR / 'test_manifest.csv'
split_manifest_path = OUTPUT_DIR / 'split_manifest.csv'
class_summary_path = OUTPUT_DIR / 'class_summary.csv'
split_summary_path = OUTPUT_DIR / 'split_summary.csv'
metadata_path = OUTPUT_DIR / 'stage3_metadata.json'

clean_columns = ['image_path', 'class_name', 'class_id', 'xml_label', 'source_folder', 'folder_matches_xml']
clean_df[clean_columns].sort_values(['class_id', 'image_path']).to_csv(clean_manifest_path, index=False)
excluded_df[['image_path', 'image_name', 'source_folder', 'reason', 'xml_labels']].sort_values(['reason', 'image_path']).to_csv(excluded_manifest_path, index=False)
train_df[clean_columns].sort_values(['class_id', 'image_path']).to_csv(train_manifest_path, index=False)
val_df[clean_columns].sort_values(['class_id', 'image_path']).to_csv(val_manifest_path, index=False)
test_df[clean_columns].sort_values(['class_id', 'image_path']).to_csv(test_manifest_path, index=False)
split_df[clean_columns + ['split']].sort_values(['split', 'class_id', 'image_path']).to_csv(split_manifest_path, index=False)
class_summary_df.to_csv(class_summary_path, index=False)
split_summary_df.to_csv(split_summary_path, index=False)

metadata = {
    'total_images': int(len(records_df)),
    'single_class_images': int(len(clean_df)),
    'excluded_images': int(len(excluded_df)),
    'train_count': int(len(train_df)),
    'val_count': int(len(val_df)),
    'test_count': int(len(test_df)),
    'random_seed': RANDOM_SEED,
}

with open(metadata_path, 'w', encoding='utf-8') as fp:
    json.dump(metadata, fp, ensure_ascii=False, indent=2)

print('Сохранено в:', OUTPUT_DIR)
for path in [clean_manifest_path, excluded_manifest_path, train_manifest_path, val_manifest_path, test_manifest_path, split_manifest_path, class_summary_path, split_summary_path, metadata_path]:
    print('-', path.name)

## Итог этапа 3

После выполнения ноутбука получаем:
- понятное описание структуры датасета;
- таблицу разметки для классификации;
- анализ распределения классов;
- примеры изображений;
- разбиение на `train / val / test`;
- вывод, что данные подготовлены к 4 этапу.